In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score

In [30]:
class SafetyMLP(nn.Module):
    def __init__(self, input_size):
        super(SafetyMLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.6),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.layers(x)

In [31]:
df_text = pd.read_csv('../../../data/cleaned_data.csv').dropna(subset=['cleaned_statement', 'label'])
df_emotions = pd.read_csv('../../../data/features_for_model.csv').dropna()

min_size = min(len(df_text), len(df_emotions))

X_text = df_text['cleaned_statement'].iloc[:min_size].reset_index(drop=True)
y = df_text['label'].iloc[:min_size].reset_index(drop=True)
X_emotions = df_emotions[['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']].iloc[:min_size].reset_index(drop=True)

# Sprawdzenie dla Ciebie:
print(f"Załadowano {min_size} wierszy.")
print(f"Kształt X_text: {X_text.shape}, Kształt X_emotions: {X_emotions.shape}")

Załadowano 31993 wierszy.
Kształt X_text: (31993,), Kształt X_emotions: (31993, 6)


In [32]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 3), min_df=2, stop_words='english')
X_tfidf = tfidf.fit_transform(X_text).toarray()
X_combined = np.hstack([X_tfidf, X_emotions.values])

X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train.values).view(-1, 1)),
    batch_size=32, shuffle=True
)

test_tensor = torch.FloatTensor(X_test_scaled)
test_y_tensor = torch.FloatTensor(y_test.values).view(-1, 1)

In [33]:
model = SafetyMLP(X_train_scaled.shape[1])
criterion = nn.BCELoss()
smoothing = 0.1
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

best_loss = float('inf')

for epoch in range(50):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()

        alpha = 0.1
        smoothed_y = batch_y * (1 - alpha) + (alpha / 2)

        loss = criterion(model(batch_x), smoothed_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        test_loss = criterion(model(test_tensor), test_y_tensor)

        scheduler.step(test_loss)

        if test_loss < best_loss:
            best_loss = test_loss
            torch.save(model.state_dict(), '../local_models/safety_mlp.pth')
            print(f"Nowy najlepszy model zapisany! (Loss: {test_loss:.4f})")

preds = (model(test_tensor) > 0.5).float()
f1 = f1_score(test_y_tensor.numpy(), preds.numpy())
print(f1)

Nowy najlepszy model zapisany! (Loss: 0.2091)
0.9382251489495139


In [34]:
os.makedirs('../local_models', exist_ok=True)
torch.save(model.state_dict(), '../local_models/safety_mlp.pth')
joblib.dump(scaler, '../local_models/scaler.pkl')
joblib.dump(tfidf, '../local_models/tfidf_vectorizer.pkl')
print("Model i artefakty zapisane w backend/ml/local_models/")

Model i artefakty zapisane w backend/ml/local_models/
